# Encoder-Free VLM - Quality-Focused Colab Notebook

This notebook implements the encoder-free VLM architecture from Train Your Own Encoder-Free VLM in $100. It is configured for a free Colab T4 with a 4-bit QLoRA decoder and resumable Google Drive checkpoints.

Important: the quality configuration uses a new Qwen2.5-1.5B training run. It cannot reuse the earlier SmolLM2-135M checkpoints.


## 📍 1. Adım: GPU Kontrolü, Paket Kurulumları ve Dizin Oluşturma

In [ ]:
# Confirm the assigned GPU.

!nvidia-smi

# Clean conflicting packages (torchao in Colab causes PEFT serialization issues)

!pip uninstall -y torchao 2>/dev/null || true

# Keep Colab's CUDA-compatible PyTorch build; install only project dependencies.

!pip install -q -U 'transformers>=4.43' peft bitsandbytes accelerate datasets pyyaml gradio pillow

# Create the project layout.

!mkdir -p src/encoder_free_vlm scripts configs outputs/encoder_free_vlm_qwen15b

## 📍 2. Adım: Google Drive Bağlantısı ve Hugging Face Hub Girişi

In [ ]:
from google.colab import drive
# Colab kopmalarında checkpoint'lerin kaybolmaması için Google Drive'ı bağlayın
drive.mount('/content/drive')

# Hugging Face Token Login
from huggingface_hub import notebook_login
notebook_login()

## 📍 3. Adım: Konfigürasyon Dosyası (`configs/colab_t4.yaml`)

In [ ]:
%%writefile configs/colab_t4.yaml
# Quality-oriented free Colab T4 configuration.
# This is a new training run: do not resume a SmolLM2-135M checkpoint into it.
# Free Colab sessions can resume safely from the Drive checkpoints written every
# 250 steps.  A 1.5B decoder is deliberately used because the accompanying
# paper found that the original 135M decoder did not learn grounded vision.

model:
  base_model_name: Qwen/Qwen2.5-1.5B-Instruct
  image_token: "<|image|>"
  trust_remote_code: true
  torch_dtype: float16
  load_in_4bit: true
  gradient_checkpointing: true

vision:
  image_size: 512
  patch_size: 32
  channels: 3

data:
  dataset_name: HuggingFaceM4/FineVision
  dataset_subset: null
  # Prioritize direct question-answering over decorative text to enhance visual grounding
  dataset_subsets:
    - LLaVA_Instruct_150K
    - "sharegpt4v(llava)"
  dataset_weights: [0.8, 0.2]
  split: train
  streaming: true
  max_samples: null
  max_length: 2048
  packing: false
  pack_pool_size: 64
  pad_to_multiple_of: 8
  # Set to 0 on free Colab to prevent caching decoded images in host RAM.
  shuffle_buffer_size: 0
  min_image_correspondence: 4
  min_visual_dependency: 3

lora:
  enabled: true
  r: 16
  alpha: 32
  dropout: 0.05
  target_modules: all-linear

training:
  output_dir: outputs/encoder_free_vlm_qwen15b
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 16
  learning_rate: 4.0e-5
  # The randomly initialised image embedder needs a faster update rate than
  # the pretrained language model's LoRA adapters.
  vision_learning_rate: 3.5e-4
  weight_decay: 0.01
  warmup_ratio: 0.03
  lr_scheduler_type: cosine
  optim: paged_adamw_8bit
  max_grad_norm: 0.5
  seed: 42
  max_steps: 500
  num_train_epochs: 1
  logging_steps: 5
  save_steps: 50
  save_total_limit: 4
  fp16: true
  bf16: false
  report_to: none


## 📍 4. Adım: Konfigürasyon Dataclass Sınıfları (`src/encoder_free_vlm/config.py`)

In [ ]:
%%writefile src/encoder_free_vlm/config.py
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@dataclass
class VisionConfig:
    image_size: int = 512
    patch_size: int = 32
    channels: int = 3

    @property
    def grid_size(self) -> int:
        if self.image_size % self.patch_size != 0:
            raise ValueError("image_size must be divisible by patch_size")
        return self.image_size // self.patch_size

    @property
    def num_patches(self) -> int:
        return self.grid_size * self.grid_size

    @property
    def flattened_patch_dim(self) -> int:
        return self.channels * self.patch_size * self.patch_size


@dataclass
class ModelConfig:
    base_model_name: str = "HuggingFaceTB/SmolLM2-135M-Instruct"
    image_token: str = "<|image|>"
    trust_remote_code: bool = True
    torch_dtype: str = "float16"
    load_in_4bit: bool = True
    gradient_checkpointing: bool = True


@dataclass
class DataConfig:
    dataset_name: str = "HuggingFaceM4/FineVision"
    dataset_subset: str | None = "LLaVA_Instruct_150K"
    # When set, several FineVision subsets are interleaved.  ``dataset_subset``
    # is retained for backward compatibility with the original notebook.
    dataset_subsets: list[str] | None = None
    dataset_weights: list[float] | None = None
    split: str = "train"
    streaming: bool = True
    max_samples: int | None = 5000
    max_length: int = 2048
    packing: bool = False
    pack_pool_size: int = 128
    pad_to_multiple_of: int | None = 8
    shuffle_buffer_size: int = 10_000
    # FineVision exposes these scores for many subsets.  Samples without a
    # score are kept so that older subsets remain usable.
    min_image_correspondence: int | None = None
    min_visual_dependency: int | None = None


@dataclass
class LoraConfigData:
    enabled: bool = True
    r: int = 16
    alpha: int = 32
    dropout: float = 0.05
    target_modules: str | list[str] = "all-linear"


@dataclass
class TrainConfig:
    output_dir: str = "outputs/encoder_free_vlm"
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    vision_learning_rate: float | None = None
    weight_decay: float = 0.0
    warmup_ratio: float = 0.03
    lr_scheduler_type: str = "linear"
    optim: str = "adamw_torch"
    max_grad_norm: float = 1.0
    seed: int = 42
    max_steps: int = 1000
    num_train_epochs: int = 1
    logging_steps: int = 10
    save_steps: int = 500
    save_total_limit: int = 3
    fp16: bool = True
    bf16: bool = False
    report_to: str | list[str] = "none"


@dataclass
class ProjectConfig:
    model: ModelConfig = field(default_factory=ModelConfig)
    vision: VisionConfig = field(default_factory=VisionConfig)
    data: DataConfig = field(default_factory=DataConfig)
    lora: LoraConfigData = field(default_factory=LoraConfigData)
    training: TrainConfig = field(default_factory=TrainConfig)


def _merge_dataclass(instance: Any, values: dict[str, Any]) -> Any:
    for key, value in values.items():
        if not hasattr(instance, key):
            raise KeyError(f"Unknown config key: {key}")
        current = getattr(instance, key)
        if hasattr(current, "__dataclass_fields__") and isinstance(value, dict):
            _merge_dataclass(current, value)
        else:
            setattr(instance, key, value)
    return instance


def load_project_config(path: str | Path | None) -> ProjectConfig:
    cfg = ProjectConfig()
    if path is None:
        return cfg

    import yaml

    data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
    return _merge_dataclass(cfg, data)


## 📍 5. Adım: Görüntü Ön İşleme (512x512 Resize & CenterCrop) (`src/encoder_free_vlm/image_processing.py`)

In [ ]:
%%writefile src/encoder_free_vlm/image_processing.py
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image, ImageOps


def load_image(image: Any) -> Image.Image:
    if isinstance(image, Image.Image):
        return ImageOps.exif_transpose(image).convert("RGB")
    if isinstance(image, (str, Path)):
        image_str = str(image)
        if image_str.startswith(("http://", "https://")):
            import io
            import urllib.request
            req = urllib.request.Request(image_str, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as resp:
                with Image.open(io.BytesIO(resp.read())) as opened:
                    return ImageOps.exif_transpose(opened).convert("RGB")
        with Image.open(image) as opened:
            return ImageOps.exif_transpose(opened).convert("RGB")
    if isinstance(image, dict) and "path" in image:
        return load_image(image["path"])
    if isinstance(image, bytes):
        import io

        with Image.open(io.BytesIO(image)) as opened:
            return ImageOps.exif_transpose(opened).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image)!r}")


def resize_shorter_side(image: Image.Image, size: int) -> Image.Image:
    width, height = image.size
    if width <= 0 or height <= 0:
        raise ValueError("image has invalid dimensions")
    scale = size / min(width, height)
    new_width = max(size, round(width * scale))
    new_height = max(size, round(height * scale))
    resample = getattr(Image, "Resampling", Image).BICUBIC
    return image.resize((new_width, new_height), resample=resample)


def center_crop(image: Image.Image, size: int) -> Image.Image:
    width, height = image.size
    left = max(0, (width - size) // 2)
    top = max(0, (height - size) // 2)
    return image.crop((left, top, left + size, top + size))


def image_to_tensor(image: Image.Image) -> torch.Tensor:
    array = np.asarray(image, dtype=np.uint8)
    if array.ndim != 3 or array.shape[2] != 3:
        raise ValueError("expected an RGB image")
    tensor = torch.from_numpy(array.copy()).permute(2, 0, 1).contiguous()
    return tensor.float().div_(255.0)


def preprocess_image(image: Any, image_size: int = 512) -> torch.Tensor:
    pil_image = load_image(image)
    pil_image = resize_shorter_side(pil_image, image_size)
    pil_image = center_crop(pil_image, image_size)
    return image_to_tensor(pil_image)


## 📍 6. Adım: Döngüsüz 32x32 Yamalama (Patchify) (`src/encoder_free_vlm/patching.py`)

In [ ]:
%%writefile src/encoder_free_vlm/patching.py
from __future__ import annotations

import torch


def extract_flattened_patches(pixel_values: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Extract flattened non-overlapping patches with reshape/permute only."""
    if pixel_values.ndim != 4:
        raise ValueError("pixel_values must have shape (batch, channels, height, width)")

    batch_size, channels, height, width = pixel_values.shape
    if height % patch_size != 0 or width % patch_size != 0:
        raise ValueError("height and width must be divisible by patch_size")

    patches_h = height // patch_size
    patches_w = width // patch_size
    x = pixel_values.reshape(batch_size, channels, patches_h, patch_size, patches_w, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)
    return x.reshape(batch_size, patches_h * patches_w, channels * patch_size * patch_size)


## 📍 7. Adım: Vision Embedder Modülü (`src/encoder_free_vlm/embedder.py`)

In [ ]:
%%writefile src/encoder_free_vlm/embedder.py
from __future__ import annotations

import torch
from torch import nn

from .config import VisionConfig
from .patching import extract_flattened_patches


class VisionPatchEmbedder(nn.Module):
    """Tiny encoder-free image embedder used before the decoder."""

    def __init__(self, vision_config: VisionConfig, hidden_size: int) -> None:
        super().__init__()
        self.vision_config = vision_config
        self.hidden_size = hidden_size
        patch_dim = vision_config.flattened_patch_dim
        grid_size = vision_config.grid_size

        self.ln1 = nn.LayerNorm(patch_dim)
        self.fc = nn.Linear(patch_dim, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.y_pos_emb = nn.Parameter(torch.zeros(1, grid_size, hidden_size))
        self.x_pos_emb = nn.Parameter(torch.zeros(1, grid_size, hidden_size))
        self.ln3 = nn.LayerNorm(hidden_size)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.y_pos_emb, mean=0.0, std=0.02)
        nn.init.normal_(self.x_pos_emb, mean=0.0, std=0.02)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        _, _, height, width = pixel_values.shape
        patch_size = self.vision_config.patch_size
        patches_h = height // patch_size
        patches_w = width // patch_size
        if patches_h > self.vision_config.grid_size or patches_w > self.vision_config.grid_size:
            raise ValueError("input image has more patches than the configured position tables")

        x = extract_flattened_patches(pixel_values, patch_size)
        x = self.ln1(x)
        x = self.fc(x)
        x = self.ln2(x)

        row_emb = self.y_pos_emb[:, :patches_h, :]
        col_emb = self.x_pos_emb[:, :patches_w, :]
        pos = (row_emb.unsqueeze(2) + col_emb.unsqueeze(1)).reshape(1, patches_h * patches_w, -1)
        x = x + pos.to(dtype=x.dtype, device=x.device)
        return self.ln3(x)


## 📍 8. Adım: Tokenizasyon ve Prompt Maskeleme (`src/encoder_free_vlm/tokenization.py`)

In [ ]:
%%writefile src/encoder_free_vlm/tokenization.py
from __future__ import annotations

from typing import Any


def ensure_image_token(tokenizer: Any, image_token: str = "<|image|>") -> int:
    vocab = tokenizer.get_vocab()
    if image_token not in vocab:
        tokenizer.add_special_tokens({"additional_special_tokens": [image_token]})
    token_id = tokenizer.convert_tokens_to_ids(image_token)
    tokenizer.image_token = image_token
    tokenizer.image_token_id = token_id
    return int(token_id)


def image_token_prefix(image_token: str, num_patches: int) -> str:
    return image_token * num_patches


def normalize_turns_from_messages(messages: list[dict[str, Any]]) -> list[dict[str, str]]:
    turns: list[dict[str, str]] = []
    pending_user: str | None = None
    for message in messages:
        role = str(message.get("role") or message.get("from") or "").lower()
        content = message.get("content", message.get("value", ""))
        if isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    text_parts.append(str(item.get("text", "")))
                elif isinstance(item, str):
                    text_parts.append(item)
            content = "\n".join(text_parts)
        content = str(content)
        if role in {"human", "user"}:
            pending_user = content
        elif role in {"gpt", "assistant", "model"} and pending_user is not None:
            turns.append({"user": pending_user, "assistant": content})
            pending_user = None
    return turns


def render_chat(
    tokenizer: Any,
    turns: list[dict[str, str]],
    image_token: str,
    num_patches: int,
    system_prompt: str | None = None,
    add_generation_prompt: bool = False,
) -> str:
    messages: list[dict[str, str]] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    for index, turn in enumerate(turns):
        user_content = turn["user"]
        if index == 0:
            user_content = image_token_prefix(image_token, num_patches) + user_content
        messages.append({"role": "user", "content": user_content})
        assistant = turn.get("assistant")
        if assistant is not None:
            messages.append({"role": "assistant", "content": assistant})

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    eos = tokenizer.eos_token or ""
    chunks = []
    for message in messages:
        chunks.append(f"{message['role']}: {message['content']}{eos}")
    if add_generation_prompt:
        chunks.append("assistant: ")
    return "\n".join(chunks)


def encode_turns(
    tokenizer: Any,
    turns: list[dict[str, str]],
    image_token: str,
    num_patches: int,
    max_length: int,
    system_prompt: str | None = None,
    add_generation_prompt: bool = False,
) -> list[int]:
    text = render_chat(
        tokenizer=tokenizer,
        turns=turns,
        image_token=image_token,
        num_patches=num_patches,
        system_prompt=system_prompt,
        add_generation_prompt=add_generation_prompt,
    )
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_length,
    )
    return list(encoded["input_ids"])


def encode_turns_with_labels(
    tokenizer: Any,
    turns: list[dict[str, str]],
    image_token: str,
    num_patches: int,
    max_length: int,
    system_prompt: str | None = None,
    mask_user_prompt: bool = True,
    ignore_index: int = -100,
) -> tuple[list[int], list[int]]:
    text = render_chat(
        tokenizer=tokenizer,
        turns=turns,
        image_token=image_token,
        num_patches=num_patches,
        system_prompt=system_prompt,
        add_generation_prompt=False,
    )
    if not mask_user_prompt:
        input_ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
        )["input_ids"]
        labels = list(input_ids)
    else:
        # Training on every user turn makes the model learn to continue a
        # conversation as the *user*.  Supervise only answer text instead.
        # Offset mappings keep this correct for multi-turn chat templates.
        answer_spans: list[tuple[int, int]] = []
        search_start = 0
        for turn in turns:
            answer = turn.get("assistant")
            if not answer:
                continue
            start = text.find(answer, search_start)
            if start < 0:
                answer_spans = []
                break
            end = start + len(answer)
            answer_spans.append((start, end))
            search_start = end

        try:
            if not answer_spans:
                raise ValueError("could not locate assistant text in the rendered chat")
            encoded = tokenizer(
                text,
                add_special_tokens=False,
                truncation=True,
                max_length=max_length,
                return_offsets_mapping=True,
            )
            input_ids = list(encoded["input_ids"])
            offsets = list(encoded["offset_mapping"])
            labels = [ignore_index] * len(input_ids)
            for index, (token_id, offset) in enumerate(zip(input_ids, offsets)):
                start, end = offset
                if end > start and any(start < span_end and end > span_start for span_start, span_end in answer_spans):
                    labels[index] = token_id
        except (TypeError, KeyError, ValueError):
            # Slow tokenizers may not provide offsets.  Preserve the original
            # first-turn masking as a safe fallback rather than failing a run.
            input_ids = encode_turns(
                tokenizer=tokenizer,
                turns=turns,
                image_token=image_token,
                num_patches=num_patches,
                max_length=max_length,
                system_prompt=system_prompt,
                add_generation_prompt=False,
            )
            labels = list(input_ids)
            prompt_text = render_chat(
                tokenizer=tokenizer,
                turns=[{"user": turns[0]["user"], "assistant": None}],
                image_token=image_token,
                num_patches=num_patches,
                system_prompt=system_prompt,
                add_generation_prompt=True,
            )
            prompt_ids = tokenizer(
                prompt_text,
                add_special_tokens=False,
                truncation=True,
                max_length=max_length,
            )["input_ids"]
            for i in range(min(len(prompt_ids), len(input_ids))):
                labels[i] = ignore_index

    # Also ensure any image_token_id is explicitly set to ignore_index
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    for i in range(len(labels)):
        if input_ids[i] == image_token_id:
            labels[i] = ignore_index

    return input_ids, labels


## 📍 9. Adım: EncoderFreeVLM Model Sınıfı (`src/encoder_free_vlm/model.py`)

In [ ]:
%%writefile src/encoder_free_vlm/model.py
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
from typing import Any

import torch
from torch import nn

from .config import VisionConfig
from .embedder import VisionPatchEmbedder


def decoder_hidden_size(decoder: nn.Module) -> int:
    config = getattr(decoder, "config", None)
    for name in ("hidden_size", "n_embd", "d_model"):
        value = getattr(config, name, None)
        if value is not None:
            return int(value)
    embeddings = decoder.get_input_embeddings()
    return int(embeddings.embedding_dim)


class EncoderFreeVLM(nn.Module):
    def __init__(
        self,
        decoder: nn.Module,
        vision_config: VisionConfig,
        image_token_id: int,
        pad_token_id: int | None,
    ) -> None:
        super().__init__()
        self.decoder = decoder
        self.vision_config = vision_config
        self.image_token_id = int(image_token_id)
        self.pad_token_id = None if pad_token_id is None else int(pad_token_id)
        hidden_size = decoder_hidden_size(decoder)
        self.vision_embedder = VisionPatchEmbedder(vision_config, hidden_size)
        self.connector = nn.Linear(hidden_size, hidden_size)

    @property
    def device(self) -> torch.device:
        return next(self.parameters()).device

    def get_input_embeddings(self) -> nn.Module:
        return self.decoder.get_input_embeddings()

    def encode_images(self, pixel_values: torch.Tensor) -> torch.Tensor:
        param = next(self.vision_embedder.parameters())
        pixel_values = pixel_values.to(device=param.device, dtype=param.dtype)
        image_embeds = self.vision_embedder(pixel_values)
        return self.connector(image_embeds)

    def build_inputs_embeds(
        self,
        input_ids: torch.Tensor,
        pixel_values: torch.Tensor,
    ) -> torch.Tensor:
        input_ids = input_ids.to(device=self.device)
        token_embeds = self.get_input_embeddings()(input_ids)
        image_embeds = self.encode_images(pixel_values)

        mask = input_ids.eq(self.image_token_id)
        flat_image_embeds = image_embeds.reshape(-1, image_embeds.shape[-1])
        expected_slots = flat_image_embeds.shape[0]
        actual_slots = int(mask.sum().item())
        if actual_slots != expected_slots:
            raise ValueError(
                f"image-token slot mismatch: input has {actual_slots}, images need {expected_slots}"
            )

        combined = token_embeds.clone()
        combined[mask] = flat_image_embeds.to(combined.dtype)
        return combined

    def forward(
        self,
        input_ids: torch.Tensor,
        pixel_values: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        use_cache: bool | None = None,
        **kwargs: Any,
    ) -> Any:
        inputs_embeds = self.build_inputs_embeds(input_ids, pixel_values)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=self.device)
        if labels is not None:
            labels = labels.to(device=self.device)
        return self.decoder(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=use_cache,
            **kwargs,
        )

    def save_pretrained(self, path: str | Path) -> None:
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        decoder_path = path / "decoder"
        if hasattr(self.decoder, "save_pretrained"):
            self.decoder.save_pretrained(decoder_path)
        torch.save(
            {
                "vision_config": asdict(self.vision_config),
                "image_token_id": self.image_token_id,
                "pad_token_id": self.pad_token_id,
                "vision_embedder": self.vision_embedder.state_dict(),
                "connector": self.connector.state_dict(),
            },
            path / "vision_adapter.pt",
        )

    def gradient_checkpointing_enable(self, **kwargs: Any) -> None:
        if hasattr(self.decoder, "gradient_checkpointing_enable"):
            self.decoder.gradient_checkpointing_enable(**kwargs)
        if hasattr(getattr(self.decoder, "config", None), "use_cache"):
            self.decoder.config.use_cache = False

    def gradient_checkpointing_disable(self) -> None:
        if hasattr(self.decoder, "gradient_checkpointing_disable"):
            self.decoder.gradient_checkpointing_disable()

    def enable_input_require_grads(self) -> None:
        if hasattr(self.decoder, "enable_input_require_grads"):
            self.decoder.enable_input_require_grads()

    def load_vision_adapter(self, path: str | Path, strict: bool = True) -> None:
        path = Path(path)
        try:
            state = torch.load(path / "vision_adapter.pt", map_location="cpu", weights_only=False)
        except TypeError:
            state = torch.load(path / "vision_adapter.pt", map_location="cpu")
        self.vision_embedder.load_state_dict(state["vision_embedder"], strict=strict)
        self.connector.load_state_dict(state["connector"], strict=strict)


## 📍 10. Adım: Loss Maskeleme Modülü (`src/encoder_free_vlm/loss.py`)

In [ ]:
%%writefile src/encoder_free_vlm/loss.py
from __future__ import annotations

import torch
import torch.nn.functional as F


def build_labels(
    input_ids: torch.Tensor,
    image_token_id: int,
    pad_token_id: int | None,
    ignore_index: int = -100,
) -> torch.Tensor:
    labels = input_ids.clone()
    labels[labels == image_token_id] = ignore_index
    if pad_token_id is not None:
        labels[labels == pad_token_id] = ignore_index
    return labels


def next_token_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    ignore_index: int = -100,
) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=ignore_index,
    )


## 📍 11. Adım: PyTorch Dataset, Knapsack ve Collator (`src/encoder_free_vlm/data.py`)

In [ ]:
%%writefile src/encoder_free_vlm/data.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, Iterator

import torch
from torch.utils.data import Dataset, IterableDataset

from .config import VisionConfig
from .image_processing import preprocess_image
from .loss import build_labels
from .tokenization import encode_turns_with_labels, normalize_turns_from_messages


@dataclass
class VisionTextSample:
    image: Any
    turns: list[dict[str, str]]


def _minimum_score(value: Any) -> int | None:
    """Return the lowest quality score, accepting FineVision's list fields."""
    if value is None:
        return None
    if isinstance(value, (list, tuple)):
        values = [item for item in (_minimum_score(item) for item in value) if item is not None]
        return min(values) if values else None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def passes_quality_filter(
    raw: dict[str, Any],
    min_image_correspondence: int | None = None,
    min_visual_dependency: int | None = None,
) -> bool:
    """Keep visually grounded examples and gracefully support unscored subsets."""
    checks = (
        ("image_correspondence_min", min_image_correspondence),
        ("visual_dependency_min", min_visual_dependency),
    )
    for field, threshold in checks:
        if threshold is None:
            continue
        score = _minimum_score(raw.get(field))
        if score is not None and score < threshold:
            return False
    return True


def normalize_sample(raw: dict[str, Any]) -> VisionTextSample:
    image = raw.get("image")
    if image is None and raw.get("images"):
        images = raw["images"]
        image = images[0] if isinstance(images, (list, tuple)) else images
    if image is None and raw.get("image_path"):
        image = raw["image_path"]
    if image is None:
        raise KeyError("sample does not contain an image")

    if "texts" in raw and isinstance(raw["texts"], list):
        turns = []
        for turn in raw["texts"]:
            if "user" in turn and "assistant" in turn:
                turns.append({"user": str(turn["user"]), "assistant": str(turn["assistant"])})
        if turns:
            return VisionTextSample(image=image, turns=turns)

    if "conversations" in raw:
        turns = normalize_turns_from_messages(raw["conversations"])
        if turns:
            return VisionTextSample(image=image, turns=turns)

    if "messages" in raw:
        turns = normalize_turns_from_messages(raw["messages"])
        if turns:
            return VisionTextSample(image=image, turns=turns)

    if "question" in raw and "answer" in raw:
        return VisionTextSample(
            image=image,
            turns=[{"user": str(raw["question"]), "assistant": str(raw["answer"])}],
        )

    if "caption" in raw:
        return VisionTextSample(
            image=image,
            turns=[{"user": "Describe this image.", "assistant": str(raw["caption"])}],
        )

    raise KeyError("sample does not contain a supported conversation format")


class VisionTextDataset(Dataset):
    def __init__(
        self,
        samples: list[dict[str, Any]],
        tokenizer: Any,
        vision_config: VisionConfig,
        image_token: str,
        max_length: int,
        system_prompt: str | None = None,
        mask_user_prompt: bool = True,
        min_image_correspondence: int | None = None,
        min_visual_dependency: int | None = None,
    ) -> None:
        self.samples = samples
        self.tokenizer = tokenizer
        self.vision_config = vision_config
        self.image_token = image_token
        self.max_length = max_length
        self.system_prompt = system_prompt
        self.mask_user_prompt = mask_user_prompt
        self.min_image_correspondence = min_image_correspondence
        self.min_visual_dependency = min_visual_dependency

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        raw = self.samples[index]
        if not passes_quality_filter(
            raw,
            min_image_correspondence=self.min_image_correspondence,
            min_visual_dependency=self.min_visual_dependency,
        ):
            raise ValueError("sample did not satisfy the configured visual-quality thresholds")
        return encode_sample(
            raw,
            self.tokenizer,
            self.vision_config,
            self.image_token,
            self.max_length,
            system_prompt=self.system_prompt,
            mask_user_prompt=self.mask_user_prompt,
        )


class VisionTextIterableDataset(IterableDataset):
    def __init__(
        self,
        samples: Iterable[dict[str, Any]],
        tokenizer: Any,
        vision_config: VisionConfig,
        image_token: str,
        max_length: int,
        system_prompt: str | None = None,
        mask_user_prompt: bool = True,
        min_image_correspondence: int | None = None,
        min_visual_dependency: int | None = None,
    ) -> None:
        self.samples = samples
        self.tokenizer = tokenizer
        self.vision_config = vision_config
        self.image_token = image_token
        self.max_length = max_length
        self.system_prompt = system_prompt
        self.mask_user_prompt = mask_user_prompt
        self.min_image_correspondence = min_image_correspondence
        self.min_visual_dependency = min_visual_dependency

    def __iter__(self) -> Iterator[dict[str, torch.Tensor]]:
        for raw in self.samples:
            try:
                if not passes_quality_filter(
                    raw,
                    min_image_correspondence=self.min_image_correspondence,
                    min_visual_dependency=self.min_visual_dependency,
                ):
                    continue
                yield encode_sample(
                    raw,
                    self.tokenizer,
                    self.vision_config,
                    self.image_token,
                    self.max_length,
                    system_prompt=self.system_prompt,
                    mask_user_prompt=self.mask_user_prompt,
                )
            except Exception:
                continue


def encode_sample(
    raw: dict[str, Any],
    tokenizer: Any,
    vision_config: VisionConfig,
    image_token: str,
    max_length: int,
    system_prompt: str | None = None,
    mask_user_prompt: bool = True,
) -> dict[str, torch.Tensor]:
    sample = normalize_sample(raw)
    pixel_values = preprocess_image(sample.image, vision_config.image_size)
    input_ids, labels = encode_turns_with_labels(
        tokenizer=tokenizer,
        turns=sample.turns,
        image_token=image_token,
        num_patches=vision_config.num_patches,
        max_length=max_length,
        system_prompt=system_prompt,
        mask_user_prompt=mask_user_prompt,
    )
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    actual_image_tokens = sum(1 for token_id in input_ids if token_id == image_token_id)
    if actual_image_tokens != vision_config.num_patches:
        raise ValueError(
            f"expected {vision_config.num_patches} image tokens, got {actual_image_tokens}"
        )
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "pixel_values": pixel_values,
    }


class KnapsackPacker(IterableDataset):
    """Greedy packing wrapper for pre-tokenized one-image samples."""

    def __init__(
        self,
        dataset: Iterable[dict[str, torch.Tensor]],
        max_length: int,
        pool_size: int = 128,
    ) -> None:
        self.dataset = dataset
        self.max_length = max_length
        self.pool_size = pool_size

    def __iter__(self) -> Iterator[dict[str, torch.Tensor]]:
        iterator = iter(self.dataset)
        while True:
            pool = []
            try:
                for _ in range(self.pool_size):
                    sample = next(iterator)
                    if sample["input_ids"].numel() <= self.max_length:
                        pool.append(sample)
            except StopIteration:
                pass

            if not pool:
                return

            yield from self._pack_pool(pool)

    def _pack_pool(self, pool: list[dict[str, torch.Tensor]]) -> Iterator[dict[str, torch.Tensor]]:
        pool.sort(key=lambda item: item["input_ids"].numel(), reverse=True)
        knapsacks: list[dict[str, Any]] = []
        for sample in pool:
            length = sample["input_ids"].numel()
            placed = False
            for sack in knapsacks:
                if sack["remaining"] >= length:
                    sack["input_ids"].append(sample["input_ids"])
                    if "labels" in sample:
                        sack["labels"].append(sample["labels"])
                    sack["pixel_values"].append(sample["pixel_values"])
                    sack["remaining"] -= length
                    placed = True
                    break
            if not placed:
                knapsack_item: dict[str, Any] = {
                    "remaining": self.max_length - length,
                    "input_ids": [sample["input_ids"]],
                    "pixel_values": [sample["pixel_values"]],
                }
                if "labels" in sample:
                    knapsack_item["labels"] = [sample["labels"]]
                knapsacks.append(knapsack_item)

        for sack in knapsacks:
            item = {
                "input_ids": torch.cat(sack["input_ids"], dim=0),
                "pixel_values": torch.stack(sack["pixel_values"], dim=0),
            }
            if "labels" in sack:
                item["labels"] = torch.cat(sack["labels"], dim=0)
            yield item


class DataCollatorForVisionText:
    def __init__(
        self,
        pad_token_id: int,
        image_token_id: int,
        pad_to_multiple_of: int | None = None,
        ignore_index: int = -100,
    ) -> None:
        self.pad_token_id = pad_token_id
        self.image_token_id = image_token_id
        self.pad_to_multiple_of = pad_to_multiple_of
        self.ignore_index = ignore_index

    def __call__(self, features: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:
        max_length = max(feature["input_ids"].numel() for feature in features)
        if self.pad_to_multiple_of:
            multiple = self.pad_to_multiple_of
            max_length = ((max_length + multiple - 1) // multiple) * multiple

        batch_ids = []
        batch_labels = []
        batch_attention = []
        image_tensors = []

        has_precomputed_labels = "labels" in features[0]

        for feature in features:
            ids = feature["input_ids"]
            pad_length = max_length - ids.numel()

            if pad_length > 0:
                pad_tokens = torch.full((pad_length,), self.pad_token_id, dtype=ids.dtype)
                padded_ids = torch.cat([ids, pad_tokens], dim=0)
            else:
                padded_ids = ids

            batch_ids.append(padded_ids)
            batch_attention.append((padded_ids != self.pad_token_id).long())

            if has_precomputed_labels:
                lbls = feature["labels"]
                if pad_length > 0:
                    pad_lbls = torch.full((pad_length,), self.ignore_index, dtype=lbls.dtype)
                    padded_lbls = torch.cat([lbls, pad_lbls], dim=0)
                else:
                    padded_lbls = lbls
                batch_labels.append(padded_lbls)

            pixels = feature["pixel_values"]
            if pixels.ndim == 3:
                image_tensors.append(pixels)
            elif pixels.ndim == 4:
                image_tensors.extend(list(pixels))
            else:
                raise ValueError("pixel_values must be (C,H,W) or (N,C,H,W)")

        input_ids = torch.stack(batch_ids, dim=0)
        attention_mask = torch.stack(batch_attention, dim=0)

        if has_precomputed_labels:
            labels = torch.stack(batch_labels, dim=0)
        else:
            labels = build_labels(input_ids, self.image_token_id, self.pad_token_id, self.ignore_index)

        pixel_values = torch.stack(image_tensors, dim=0)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "pixel_values": pixel_values,
        }


## 📍 12. Adım: 4-Bit QLoRA ve Model Yükleme Araçları (`src/encoder_free_vlm/train_utils.py`)

In [ ]:
%%writefile src/encoder_free_vlm/train_utils.py
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import torch

from .config import LoraConfigData, ModelConfig, VisionConfig
from .model import EncoderFreeVLM
from .tokenization import ensure_image_token


def torch_dtype_from_name(name: str) -> torch.dtype:
    normalized = name.lower()
    if normalized in {"float16", "fp16", "half"}:
        return torch.float16
    if normalized in {"bfloat16", "bf16"}:
        return torch.bfloat16
    if normalized in {"float32", "fp32"}:
        return torch.float32
    raise ValueError(f"unsupported torch dtype: {name}")


def load_tokenizer_and_decoder(
    model_config: ModelConfig,
    lora_config: LoraConfigData | None = None,
) -> tuple[Any, torch.nn.Module, int, int]:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        model_config.base_model_name,
        trust_remote_code=model_config.trust_remote_code,
    )
    image_token_id = ensure_image_token(tokenizer, model_config.image_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    pad_token_id = int(tokenizer.pad_token_id)

    dtype = torch_dtype_from_name(model_config.torch_dtype)
    model_kwargs: dict[str, Any] = {
        "trust_remote_code": model_config.trust_remote_code,
        "torch_dtype": dtype,
    }
    if model_config.load_in_4bit:
        from transformers import BitsAndBytesConfig

        compute_dtype = torch.bfloat16 if dtype == torch.bfloat16 else torch.float16
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"

    decoder = AutoModelForCausalLM.from_pretrained(model_config.base_model_name, **model_kwargs)
    decoder.resize_token_embeddings(len(tokenizer))

    if model_config.gradient_checkpointing and hasattr(decoder, "gradient_checkpointing_enable"):
        decoder.gradient_checkpointing_enable()
        if hasattr(decoder.config, "use_cache"):
            decoder.config.use_cache = False

    if lora_config and lora_config.enabled:
        from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

        if model_config.load_in_4bit:
            decoder = prepare_model_for_kbit_training(decoder)
        peft_config = LoraConfig(
            r=lora_config.r,
            lora_alpha=lora_config.alpha,
            lora_dropout=lora_config.dropout,
            target_modules=lora_config.target_modules,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        decoder = get_peft_model(decoder, peft_config)

    return tokenizer, decoder, image_token_id, pad_token_id


def build_vlm(
    model_config: ModelConfig,
    vision_config: VisionConfig,
    lora_config: LoraConfigData | None = None,
) -> tuple[Any, EncoderFreeVLM]:
    tokenizer, decoder, image_token_id, pad_token_id = load_tokenizer_and_decoder(
        model_config,
        lora_config,
    )
    model = EncoderFreeVLM(
        decoder=decoder,
        vision_config=vision_config,
        image_token_id=image_token_id,
        pad_token_id=pad_token_id,
    )
    return tokenizer, model


def find_latest_checkpoint(output_dir: str | Path) -> str | None:
    output_path = Path(output_dir)
    search_paths = [output_path]

    # Google Drive auto-resume support in Colab
    drive_base = Path("/content/drive/MyDrive")
    if drive_base.exists():
        search_paths.append(drive_base / output_path.name)
        search_paths.append(drive_base / output_dir)

    pattern = re.compile(r"checkpoint-(\d+)$")
    candidates: list[tuple[int, Path]] = []
    for path in search_paths:
        if not path.exists():
            continue
        for child in path.iterdir():
            match = pattern.match(child.name)
            if child.is_dir() and match:
                # Ensure checkpoint has at least vision_adapter.pt, trainer_state.json, or decoder
                if (child / "vision_adapter.pt").exists() or (child / "trainer_state.json").exists() or (child / "decoder").exists():
                    candidates.append((int(match.group(1)), child))

    if not candidates:
        return None
    return str(max(candidates, key=lambda item: item[0])[1])


def mark_only_vlm_trainable(model: EncoderFreeVLM) -> None:
    for parameter in model.vision_embedder.parameters():
        parameter.requires_grad = True
    for parameter in model.connector.parameters():
        parameter.requires_grad = True


## 📍 13. Adım: Çıkarım (Inference) Fonksiyonu (`src/encoder_free_vlm/generation.py`)

In [ ]:
%%writefile src/encoder_free_vlm/generation.py
from __future__ import annotations

from typing import Any

import torch

from .config import VisionConfig
from .image_processing import preprocess_image
from .model import EncoderFreeVLM
from .tokenization import encode_turns


@torch.inference_mode()
def generate_answer(
    model: EncoderFreeVLM,
    tokenizer: Any,
    image: Any,
    prompt: str,
    vision_config: VisionConfig,
    image_token: str,
    system_prompt: str | None = None,
    max_length: int = 2048,
    max_new_tokens: int = 96,
    temperature: float = 0.0,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    model.eval()
    turns = [{"user": prompt, "assistant": None}]
    input_ids = encode_turns(
        tokenizer=tokenizer,
        turns=turns,
        image_token=image_token,
        num_patches=vision_config.num_patches,
        max_length=max_length,
        system_prompt=system_prompt,
        add_generation_prompt=True,
    )
    device = model.device
    input_ids_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids_tensor)
    pixel_values = preprocess_image(image, vision_config.image_size).unsqueeze(0)
    inputs_embeds = model.build_inputs_embeds(input_ids_tensor, pixel_values)

    # In Qwen2.5, both <|im_end|> (151645) and <|endoftext|> (151643) act as stop tokens.
    # Passing both to eos_token_id ensures min_new_tokens blocks BOTH during initial generation.
    eos_token_id: list[int] = []
    for end_name in ["<|im_end|>", "<|endoftext|>"]:
        try:
            tid = tokenizer.convert_tokens_to_ids(end_name)
            if tid is not None and isinstance(tid, int) and tid > 0 and tid not in eos_token_id:
                eos_token_id.append(tid)
        except Exception:
            pass
    if not eos_token_id and getattr(tokenizer, "eos_token_id", None) is not None:
        eos_token_id = [tokenizer.eos_token_id]

    bad_words_ids: list[list[int]] = []
    for banned_tok in [
        image_token,
        "<|image|>",
        "<|im_start|>",
        "<|image_pad|>",
        "<|vision_start|>",
        "<|vision_end|>",
        "<|vision_pad|>",
    ]:
        try:
            tid = tokenizer.convert_tokens_to_ids(banned_tok)
            if tid is not None and isinstance(tid, int) and tid > 0 and [tid] not in bad_words_ids:
                bad_words_ids.append([tid])
        except Exception:
            pass

    do_sample = temperature is not None and temperature > 0.05

    generation_kwargs = {
        "input_ids": input_ids_tensor,
        "inputs_embeds": inputs_embeds,
        "attention_mask": attention_mask,
        "max_new_tokens": max_new_tokens,
        "min_new_tokens": max(12, min(max_new_tokens, 12)),
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id if getattr(tokenizer, "pad_token_id", None) is not None else eos_token_id[0],
        "eos_token_id": eos_token_id if len(eos_token_id) > 1 else eos_token_id[0],
        "bad_words_ids": bad_words_ids if bad_words_ids else None,
    }
    if repetition_penalty and repetition_penalty > 1.0:
        generation_kwargs["repetition_penalty"] = repetition_penalty
    if do_sample:
        generation_kwargs["temperature"] = max(temperature, 0.1)
        generation_kwargs["top_p"] = top_p

    generated = model.decoder.generate(**generation_kwargs)
    new_tokens = generated[:, input_ids_tensor.shape[1] :]
    decoded = tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()
    if not decoded:
        # If skip_special_tokens stripped everything, try decoding with special tokens to inspect
        decoded = tokenizer.decode(new_tokens[0], skip_special_tokens=False).strip()
    return decoded


## 📍 14. Adım: Paket Dışa Aktarımları (`src/encoder_free_vlm/__init__.py`)

In [ ]:
%%writefile src/encoder_free_vlm/__init__.py
from .config import DataConfig, ModelConfig, TrainConfig, VisionConfig
from .embedder import VisionPatchEmbedder
from .model import EncoderFreeVLM

__all__ = [
    "DataConfig",
    "EncoderFreeVLM",
    "ModelConfig",
    "TrainConfig",
    "VisionConfig",
    "VisionPatchEmbedder",
]


## 📍 15. Adım: Smoke Test Scripti ve İleri/Geri Yayılım Doğrulaması (Hafta 1 - Gün 7)

In [ ]:
%%writefile scripts/smoke_test.py
from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import torch
from torch import nn

from encoder_free_vlm.config import VisionConfig
from encoder_free_vlm.data import DataCollatorForVisionText
from encoder_free_vlm.loss import build_labels
from encoder_free_vlm.model import EncoderFreeVLM
from encoder_free_vlm.patching import extract_flattened_patches


class ToyDecoder(nn.Module):
    def __init__(self, vocab_size: int = 64, hidden_size: int = 32) -> None:
        super().__init__()
        self.config = SimpleNamespace(hidden_size=hidden_size, vocab_size=vocab_size, use_cache=False)
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def get_input_embeddings(self) -> nn.Module:
        return self.embed

    def forward(self, inputs_embeds, attention_mask=None, labels=None, use_cache=None, **kwargs):
        logits = self.lm_head(inputs_embeds)
        loss = None
        if labels is not None:
            from encoder_free_vlm.loss import next_token_loss

            loss = next_token_loss(logits, labels)
        return SimpleNamespace(logits=logits, loss=loss)


def main() -> None:
    vision = VisionConfig(image_size=64, patch_size=16, channels=3)
    pixels = torch.rand(2, 3, 64, 64)
    patches = extract_flattened_patches(pixels, vision.patch_size)
    assert patches.shape == (2, vision.num_patches, vision.flattened_patch_dim)

    image_token_id = 1
    pad_token_id = 0
    text_ids = torch.tensor([[5, 6, 7, 2], [8, 9, 2, 0]], dtype=torch.long)
    image_slots = torch.full((2, vision.num_patches), image_token_id, dtype=torch.long)
    input_ids = torch.cat([image_slots, text_ids], dim=1)
    attention_mask = (input_ids != pad_token_id).long()
    labels = build_labels(input_ids, image_token_id, pad_token_id)
    assert labels[:, : vision.num_patches].eq(-100).all()
    assert labels[input_ids == pad_token_id].eq(-100).all()

    model = EncoderFreeVLM(
        decoder=ToyDecoder(),
        vision_config=vision,
        image_token_id=image_token_id,
        pad_token_id=pad_token_id,
    )
    output = model(
        input_ids=input_ids,
        pixel_values=pixels,
        attention_mask=attention_mask,
        labels=labels,
    )
    assert output.logits.shape[:2] == input_ids.shape
    assert output.loss is not None
    output.loss.backward()
    print("smoke test passed")


if __name__ == "__main__":
    main()


In [ ]:
# Smoke Testi Çalıştır
!python -u scripts/smoke_test.py

## 📍 16. Adım: Ana Eğitim Scripti (`scripts/train.py`)

In [ ]:
%%writefile scripts/train.py
from __future__ import annotations

import argparse
from itertools import islice
import json
from pathlib import Path
import re
import shutil
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from encoder_free_vlm.config import load_project_config
from encoder_free_vlm.data import (
    DataCollatorForVisionText,
    KnapsackPacker,
    VisionTextDataset,
    VisionTextIterableDataset,
)
from encoder_free_vlm.train_utils import build_vlm, find_latest_checkpoint, mark_only_vlm_trainable


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="configs/colab_t4.yaml")
    parser.add_argument("--no-resume", action="store_true")
    parser.add_argument("--overfit-test", action="store_true", help="Run Day 19 overfit sanity test on 100 samples")
    parser.add_argument("--system-prompt", type=str, default=None, help="System prompt for instruction training")
    return parser.parse_args()


def load_training_dataset(cfg, tokenizer, system_prompt: str | None = None):
    from datasets import concatenate_datasets, interleave_datasets, load_dataset

    subsets = cfg.data.dataset_subsets
    if not subsets:
        subsets = [cfg.data.dataset_subset] if cfg.data.dataset_subset else [None]

    datasets = []
    for subset in subsets:
        load_kwargs = {
            "path": cfg.data.dataset_name,
            "split": cfg.data.split,
            "streaming": cfg.data.streaming,
        }
        if subset:
            load_kwargs["name"] = subset
        datasets.append(load_dataset(**load_kwargs))

    if len(datasets) == 1:
        dataset = datasets[0]
    elif cfg.data.streaming:
        dataset = interleave_datasets(
            datasets,
            probabilities=cfg.data.dataset_weights,
            seed=cfg.training.seed,
            stopping_strategy="all_exhausted",
        )
    else:
        dataset = concatenate_datasets(datasets).shuffle(seed=cfg.training.seed)

    if cfg.data.streaming and cfg.data.shuffle_buffer_size:
        dataset = dataset.shuffle(
            seed=cfg.training.seed,
            buffer_size=cfg.data.shuffle_buffer_size,
        )

    if cfg.data.max_samples is not None:
        if cfg.data.streaming:
            dataset = dataset.take(cfg.data.max_samples)
        else:
            dataset = dataset.select(range(min(cfg.data.max_samples, len(dataset))))

    if cfg.data.streaming:
        encoded = VisionTextIterableDataset(
            dataset,
            tokenizer=tokenizer,
            vision_config=cfg.vision,
            image_token=cfg.model.image_token,
            max_length=cfg.data.max_length,
            system_prompt=system_prompt,
            mask_user_prompt=True,
            min_image_correspondence=cfg.data.min_image_correspondence,
            min_visual_dependency=cfg.data.min_visual_dependency,
        )
    else:
        encoded = VisionTextDataset(
            list(dataset),
            tokenizer=tokenizer,
            vision_config=cfg.vision,
            image_token=cfg.model.image_token,
            max_length=cfg.data.max_length,
            system_prompt=system_prompt,
            mask_user_prompt=True,
            min_image_correspondence=cfg.data.min_image_correspondence,
            min_visual_dependency=cfg.data.min_visual_dependency,
        )

    if cfg.data.packing:
        return KnapsackPacker(
            encoded,
            max_length=cfg.data.max_length,
            pool_size=cfg.data.pack_pool_size,
        )
    return encoded


def main() -> None:
    args = parse_args()
    cfg = load_project_config(args.config)

    if getattr(args, "overfit_test", False):
        # A streaming shuffle buffer stores complete image examples in CPU RAM.
        # Do not inherit the quality-run buffer/packing settings for this tiny
        # sanity check: on free Colab that can exhaust host RAM before the
        # first optimisation step.
        print(">>> Low-RAM overfit sanity mode: cache 8 samples, run 12 steps <<<")
        cfg.data.dataset_subsets = None
        cfg.data.dataset_subset = "LLaVA_Instruct_150K"
        cfg.data.dataset_weights = None
        cfg.data.max_samples = 64
        cfg.data.max_length = 1024
        cfg.data.packing = False
        cfg.data.shuffle_buffer_size = 0
        cfg.data.min_image_correspondence = None
        cfg.data.min_visual_dependency = None
        cfg.training.max_steps = 12
        cfg.training.gradient_accumulation_steps = 1
        cfg.training.logging_steps = 1
        cfg.training.save_steps = 1_000
        cfg.training.save_total_limit = 1
        # Starting the randomly initialised image projection too aggressively
        # can overflow FP16 gradients on a T4.  Use a short warm-up and small,
        # deliberately conservative rates for this numerical-stability check.
        cfg.training.learning_rate = 5e-5
        cfg.training.vision_learning_rate = 1e-4
        cfg.training.warmup_ratio = 0.25
        cfg.training.max_grad_norm = 0.5
        cfg.training.output_dir = "outputs/overfit_test_qwen"

    tokenizer, model = build_vlm(cfg.model, cfg.vision, cfg.lora)
    mark_only_vlm_trainable(model)

    resume_from = None
    can_resume_trainer = False
    if not args.no_resume and not getattr(args, "overfit_test", False):
        resume_from = find_latest_checkpoint(cfg.training.output_dir)

    if resume_from:
        resume_path = Path(resume_from)
        # If the latest checkpoint is on Google Drive, sync it to the local workspace
        # so Hugging Face Trainer can resume natively with fast disk access.
        local_target = Path(cfg.training.output_dir) / resume_path.name
        if resume_path.resolve() != local_target.resolve():
            print(f"Syncing checkpoint {resume_path.name} from Google Drive to local workspace...")
            local_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(resume_path, local_target, dirs_exist_ok=True)
            resume_from = str(local_target)
            resume_path = local_target

        print(f"Loading VLM checkpoint weights from: {resume_from}")
        try:
            model.load_vision_adapter(resume_from)
            decoder_adapter = resume_path / "decoder"
            if decoder_adapter.exists():
                try:
                    import peft.import_utils
                    peft.import_utils.is_torchao_available = lambda: False
                except Exception:
                    pass
                if hasattr(model.decoder, "load_adapter"):
                    model.decoder.load_adapter(str(decoder_adapter), adapter_name="default", is_trainable=True)
                else:
                    from peft import PeftModel
                    model.decoder = PeftModel.from_pretrained(model.decoder, decoder_adapter, is_trainable=True)
            print("Successfully restored VLM weights from checkpoint!")
        except Exception as exc:
            print(f"Note: Could not restore adapter weights ({exc}), starting fresh.")

        state_file = resume_path / "trainer_state.json"
        if state_file.exists():
            can_resume_trainer = True
            try:
                state_data = json.loads(state_file.read_text(encoding="utf-8"))
                saved_step = int(state_data.get("global_step", 0))
                print(f">>> Found Trainer state at step {saved_step}. Resuming training seamlessly from step {saved_step}! <<<")
                if saved_step >= cfg.training.max_steps:
                    new_max = saved_step + 250
                    print(f">>> [Notice] Saved step ({saved_step}) is >= max_steps ({cfg.training.max_steps}). Automatically extending max_steps to {new_max}! <<<")
                    cfg.training.max_steps = new_max
            except Exception as e:
                print(f"Note: Error inspecting trainer_state.json ({e})")

    train_dataset = load_training_dataset(cfg, tokenizer, system_prompt=args.system_prompt)
    if args.overfit_test:
        # Materialising a handful of already preprocessed samples makes this a
        # genuine overfit check: Trainer repeatedly sees the same examples,
        # while host-RAM use stays bounded to roughly eight image tensors.
        train_dataset = list(islice(train_dataset, 8))
        if len(train_dataset) < 8:
            raise RuntimeError("could not collect eight valid samples for the overfit sanity check")
        print(f">>> Cached {len(train_dataset)} fixed samples for the low-RAM sanity check <<<")
    collator = DataCollatorForVisionText(
        pad_token_id=tokenizer.pad_token_id,
        image_token_id=tokenizer.image_token_id,
        pad_to_multiple_of=cfg.data.pad_to_multiple_of,
    )

    from transformers import Trainer, TrainerCallback, TrainingArguments

    class SaveEncoderFreeVLMCallback(TrainerCallback):
        def on_save(self, args, state, control, model=None, **kwargs):
            ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
            if model is not None and hasattr(model, "save_pretrained"):
                model.save_pretrained(ckpt_dir)

            # Auto-backup to Google Drive if running in Colab
            drive_base = Path("/content/drive/MyDrive")
            if drive_base.exists():
                drive_parent = drive_base / Path(args.output_dir).name
                drive_target = drive_parent / f"checkpoint-{state.global_step}"
                try:
                    drive_target.mkdir(parents=True, exist_ok=True)
                    shutil.copytree(ckpt_dir, drive_target, dirs_exist_ok=True)
                    print(f"Synced checkpoint to Google Drive: {drive_target}")

                    # Clean up older Drive checkpoints according to save_total_limit
                    if args.save_total_limit is not None and args.save_total_limit > 0:
                        drive_ckpts = []
                        for p in drive_parent.glob("checkpoint-*"):
                            if p.is_dir():
                                m = re.match(r"checkpoint-(\d+)$", p.name)
                                if m:
                                    drive_ckpts.append((int(m.group(1)), p))
                        if len(drive_ckpts) > args.save_total_limit:
                            drive_ckpts.sort(key=lambda x: x[0])
                            for _, old_p in drive_ckpts[:-args.save_total_limit]:
                                try:
                                    shutil.rmtree(old_p)
                                    print(f"Cleaned up older Drive checkpoint: {old_p.name}")
                                except Exception:
                                    pass
                except Exception as exc:
                    print(f"Failed to sync checkpoint to Drive: {exc}")
            return control

    report_to = cfg.training.report_to
    if report_to == "none":
        report_to = []

    warmup_steps = int(cfg.training.max_steps * getattr(cfg.training, "warmup_ratio", 0.03))

    training_args = TrainingArguments(
        output_dir=cfg.training.output_dir,
        per_device_train_batch_size=cfg.training.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.training.gradient_accumulation_steps,
        learning_rate=cfg.training.learning_rate,
        weight_decay=cfg.training.weight_decay,
        warmup_steps=warmup_steps,
        max_steps=cfg.training.max_steps,
        num_train_epochs=cfg.training.num_train_epochs,
        logging_steps=cfg.training.logging_steps,
        save_steps=cfg.training.save_steps,
        save_total_limit=cfg.training.save_total_limit,
        fp16=cfg.training.fp16,
        report_to=report_to,
        remove_unused_columns=False,
        gradient_checkpointing=cfg.model.gradient_checkpointing,
        lr_scheduler_type=cfg.training.lr_scheduler_type,
        max_grad_norm=cfg.training.max_grad_norm,
        optim=cfg.training.optim,
        seed=cfg.training.seed,
    )

    class VLMTrainer(Trainer):
        def create_optimizer(self):
            if self.optimizer is not None:
                return self.optimizer

            visual_parameters = []
            decoder_parameters = []
            for name, parameter in self.model.named_parameters():
                if not parameter.requires_grad:
                    continue
                if name.startswith(("vision_embedder.", "connector.")):
                    visual_parameters.append(parameter)
                else:
                    decoder_parameters.append(parameter)

            parameter_groups = [
                {
                    "params": decoder_parameters,
                    "lr": cfg.training.learning_rate,
                    "weight_decay": cfg.training.weight_decay,
                },
                {
                    "params": visual_parameters,
                    "lr": cfg.training.vision_learning_rate or cfg.training.learning_rate,
                    "weight_decay": cfg.training.weight_decay,
                },
            ]
            optimizer_cls, optimizer_kwargs = self.get_optimizer_cls_and_kwargs(self.args)
            self.optimizer = optimizer_cls(parameter_groups, **optimizer_kwargs)
            return self.optimizer

        def _save(self, output_dir: str | None = None, state_dict=None):
            target_dir = Path(output_dir) if output_dir is not None else Path(self.args.output_dir)
            target_dir.mkdir(parents=True, exist_ok=True)
            if hasattr(self.model, "save_pretrained"):
                self.model.save_pretrained(target_dir)
            if hasattr(self.processing_class, "save_pretrained"):
                self.processing_class.save_pretrained(target_dir)

        def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
            target_model = model if model is not None else self.model
            ckpt_path = Path(resume_from_checkpoint)
            print(f"Restoring EncoderFreeVLM weights from: {ckpt_path}")

            # 1. Restore vision embedder & connector weights
            vision_path = ckpt_path / "vision_adapter.pt"
            if vision_path.exists() and hasattr(target_model, "load_vision_adapter"):
                try:
                    target_model.load_vision_adapter(ckpt_path)
                except Exception as exc:
                    print(f"Note: Could not restore vision adapter in Trainer ({exc})")

            # 2. Restore LoRA decoder adapter weights
            decoder_adapter = ckpt_path / "decoder"
            if decoder_adapter.exists():
                decoder = getattr(target_model, "decoder", None)
                if decoder is not None:
                    try:
                        import peft.import_utils
                        peft.import_utils.is_torchao_available = lambda: False
                    except Exception:
                        pass
                    if hasattr(decoder, "load_adapter"):
                        try:
                            decoder.load_adapter(str(decoder_adapter), adapter_name="default", is_trainable=True)
                        except Exception as exc:
                            print(f"Note: Could not load_adapter on decoder ({exc})")
                    else:
                        from peft import PeftModel
                        target_model.decoder = PeftModel.from_pretrained(decoder, decoder_adapter, is_trainable=True)

    trainer = VLMTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        callbacks=[SaveEncoderFreeVLMCallback()],
    )

    resume_checkpoint_arg = resume_from if can_resume_trainer else None
    trainer.train(resume_from_checkpoint=resume_checkpoint_arg)
    trainer.save_model(cfg.training.output_dir)


if __name__ == "__main__":
    main()


## Low-RAM Overfit Sanity Check (8 cached samples / 12 steps)


In [ ]:
# Sanity check: this verifies the new Qwen-based pipeline can optimise.
# It is not a quality benchmark; continue with the main run after it succeeds.
!python scripts/train.py --config configs/colab_t4.yaml --overfit-test


## 📍 18. Adım: Asıl QLoRA Eğitimini Başlatma (Hafta 3 - Gün 21)

In [ ]:
# Start or resume the quality-focused run. Colab sessions resume automatically
# from the latest Google Drive checkpoint in encoder_free_vlm_qwen15b.
!python scripts/train.py --config configs/colab_t4.yaml


## 📍 19. Adım: LoRA Ağırlıklarını Birleştirme Scripti (`scripts/merge_lora.py`)

In [ ]:
%%writefile scripts/merge_lora.py
from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from encoder_free_vlm.config import ModelConfig
from encoder_free_vlm.tokenization import ensure_image_token


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-model", required=True)
    parser.add_argument("--checkpoint-dir", default="outputs/encoder_free_vlm_qwen15b")
    parser.add_argument("--output-dir", default="outputs/encoder_free_vlm_qwen15b/merged")
    parser.add_argument("--image-token", default="<|image|>")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    try:
        import peft.import_utils
        peft.import_utils.is_torchao_available = lambda: False
        import peft.tuners.lora.torchao
        peft.tuners.lora.torchao.is_torchao_available = lambda: False
    except Exception:
        pass
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from encoder_free_vlm.train_utils import find_latest_checkpoint

    checkpoint_dir = Path(args.checkpoint_dir)
    if not (checkpoint_dir / "decoder").exists():
        search_path = checkpoint_dir if checkpoint_dir.is_dir() else checkpoint_dir.parent
        latest = find_latest_checkpoint(search_path)
        if latest and (Path(latest) / "decoder").exists():
            checkpoint_dir = Path(latest)
        else:
            raise FileNotFoundError(f"no valid checkpoint with 'decoder' found in {args.checkpoint_dir}")
    print(f"Using checkpoint: {checkpoint_dir}")

    tokenizer = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
    ensure_image_token(tokenizer, args.image_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype="auto",
        trust_remote_code=True,
        device_map="auto",
    )
    base.resize_token_embeddings(len(tokenizer))
    peft_model = PeftModel.from_pretrained(base, checkpoint_dir / "decoder")
    merged = peft_model.merge_and_unload()

    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(output / "decoder_merged")
    tokenizer.save_pretrained(output)

    vision_adapter = checkpoint_dir / "vision_adapter.pt"
    if vision_adapter.exists():
        shutil.copy2(vision_adapter, output / "vision_adapter.pt")

    # Keep a tiny config marker for downstream scripts and Spaces.
    (output / "model_config.txt").write_text(
        f"base_model_name={ModelConfig(base_model_name=args.base_model).base_model_name}\n",
        encoding="utf-8",
    )


if __name__ == "__main__":
    main()


In [ ]:
# Optional: merge the adapter after training.

# Automatically detects the latest saved checkpoint (or provide a specific milestone).

!python scripts/merge_lora.py \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --checkpoint-dir outputs/encoder_free_vlm_qwen15b \
  --output-dir outputs/encoder_free_vlm_qwen15b/merged

## 📍 20. Adım: Çıkarım (Inference) Scripti (`scripts/infer.py`)

In [ ]:
%%writefile scripts/infer.py
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from encoder_free_vlm.config import LoraConfigData, ModelConfig, VisionConfig
from encoder_free_vlm.generation import generate_answer
from encoder_free_vlm.train_utils import build_vlm, find_latest_checkpoint


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--checkpoint-dir", default=None)
    parser.add_argument(
        "--output-dir",
        default=None,
        help="Find the newest checkpoint in this output directory (also checks Google Drive in Colab).",
    )
    parser.add_argument("--image", required=True)
    parser.add_argument("--prompt", required=True)
    parser.add_argument("--system-prompt", default=None, help="System prompt for generation")
    parser.add_argument("--max-new-tokens", type=int, default=64)
    parser.add_argument("--image-size", type=int, default=512)
    parser.add_argument("--patch-size", type=int, default=32)
    parser.add_argument(
        "--load-in-4bit",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Use the same 4-bit loading mode as training (enabled by default).",
    )
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--top-p", type=float, default=0.9)
    parser.add_argument("--repetition-penalty", type=float, default=1.1)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.checkpoint_dir and args.output_dir:
        raise ValueError("use either --checkpoint-dir or --output-dir, not both")

    checkpoint_path = args.checkpoint_dir
    if args.output_dir:
        # Check if merged model exists first
        merged_candidate = Path(args.output_dir) / "merged"
        if not (merged_candidate / "decoder_merged").exists():
            drive_merged = Path("/content/drive/MyDrive") / Path(args.output_dir).name / "merged"
            if (drive_merged / "decoder_merged").exists():
                merged_candidate = drive_merged

        if (merged_candidate / "decoder_merged").exists():
            checkpoint_path = str(merged_candidate)
            print(f"Using merged model from: {checkpoint_path}")
        else:
            checkpoint_path = find_latest_checkpoint(args.output_dir)
            if checkpoint_path is None:
                raise FileNotFoundError(f"no checkpoint found in {args.output_dir}")
            print(f"Using newest checkpoint: {checkpoint_path}")
    checkpoint = Path(checkpoint_path) if checkpoint_path else None

    # Check if checkpoint directory contains a merged decoder
    base_model_name = args.base_model
    if checkpoint:
        merged_decoder = checkpoint / "decoder_merged"
        if merged_decoder.exists():
            base_model_name = str(merged_decoder)

    model_cfg = ModelConfig(
        base_model_name=base_model_name,
        load_in_4bit=args.load_in_4bit,
        torch_dtype="float16",
    )
    vision_cfg = VisionConfig(image_size=args.image_size, patch_size=args.patch_size)
    tokenizer, model = build_vlm(model_cfg, vision_cfg, LoraConfigData(enabled=False))
    if not getattr(tokenizer, "chat_template", None):
        try:
            from transformers import AutoTokenizer
            orig_tok = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
            tokenizer.chat_template = orig_tok.chat_template
        except Exception:
            pass

    if checkpoint:
        decoder_adapter = checkpoint / "decoder"
        if decoder_adapter.exists():
            try:
                import peft.import_utils
                peft.import_utils.is_torchao_available = lambda: False
                import peft.tuners.lora.torchao
                peft.tuners.lora.torchao.is_torchao_available = lambda: False
            except Exception:
                pass
            from peft import PeftModel

            model.decoder = PeftModel.from_pretrained(model.decoder, decoder_adapter)
        vision_path = None
        candidates = [checkpoint / "vision_adapter.pt"]
        if checkpoint.name == "merged":
            candidates.append(checkpoint.parent / "vision_adapter.pt")
            candidates.append(checkpoint.parent / "checkpoint-500" / "vision_adapter.pt")
            candidates.append(Path("/content/drive/MyDrive") / checkpoint.parent.name / "checkpoint-500" / "vision_adapter.pt")
            candidates.append(Path("/content/drive/MyDrive") / checkpoint.parent.name / "merged" / "vision_adapter.pt")
        for c in candidates:
            if c.exists():
                vision_path = c
                break
        if not vision_path and args.output_dir:
            latest = find_latest_checkpoint(args.output_dir)
            if latest and (Path(latest) / "vision_adapter.pt").exists():
                vision_path = Path(latest) / "vision_adapter.pt"

        if vision_path and vision_path.exists():
            print(f"Loading vision adapter from: {vision_path}")
            model.load_vision_adapter(vision_path.parent)
        else:
            print(f"Note: No vision_adapter.pt found in {checkpoint}; vision weights remain randomly initialized.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if not args.load_in_4bit and hasattr(model, "to"):
        model.to(device)
    else:
        model.vision_embedder.to(device)
        model.connector.to(device)

    answer = generate_answer(
        model=model,
        tokenizer=tokenizer,
        image=args.image,
        prompt=args.prompt,
        vision_config=vision_cfg,
        image_token=model_cfg.image_token,
        system_prompt=args.system_prompt,
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_p=args.top_p,
        repetition_penalty=args.repetition_penalty,
    )
    print("\n" + "=" * 25 + " MODEL CEVABI " + "=" * 25, flush=True)
    if answer:
        print(answer, flush=True)
    else:
        print("(Model boş çıktı üretti)", flush=True)
    print("=" * 64 + "\n", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# Download a held-out test image and run deterministic visual inference.
!wget -q -O sample.png https://raw.githubusercontent.com/gradio-app/gradio/main/test/test_files/bus.png

!python scripts/infer.py \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --checkpoint-dir outputs/encoder_free_vlm_qwen15b/checkpoint-500 \
  --image sample.png \
  --prompt 'What is the main color of this toy vehicle, and what kind of vehicle is it?' \
  --temperature 0.0 \
  --max-new-tokens 80

## 📍 21. Adım: Gradio Web Arayüzü Uygulaması (`app.py`)

In [ ]:
%%writefile app.py
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from encoder_free_vlm.config import LoraConfigData, ModelConfig, VisionConfig
from encoder_free_vlm.generation import generate_answer
from encoder_free_vlm.train_utils import build_vlm, find_latest_checkpoint


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--checkpoint-dir", default=None)
    parser.add_argument("--output-dir", default=None)
    parser.add_argument("--load-in-4bit", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--share", action="store_true", default=True, help="Create a public Gradio share link (essential for Colab)")
    parser.add_argument("--no-share", dest="share", action="store_false", help="Do not create a public Gradio link")
    return parser.parse_args()


def main() -> None:
    import gradio as gr

    args = parse_args()
    if args.checkpoint_dir and args.output_dir:
        raise ValueError("use either --checkpoint-dir or --output-dir, not both")
    checkpoint_path = args.checkpoint_dir
    if args.output_dir:
        # Check if merged model exists first
        merged_candidate = Path(args.output_dir) / "merged"
        if not (merged_candidate / "decoder_merged").exists():
            drive_merged = Path("/content/drive/MyDrive") / Path(args.output_dir).name / "merged"
            if (drive_merged / "decoder_merged").exists():
                merged_candidate = drive_merged

        if (merged_candidate / "decoder_merged").exists():
            checkpoint_path = str(merged_candidate)
            print(f"Using merged model from: {checkpoint_path}")
        else:
            checkpoint_path = find_latest_checkpoint(args.output_dir)
            if checkpoint_path is None:
                raise FileNotFoundError(f"no checkpoint found in {args.output_dir}")
            print(f"Using newest checkpoint: {checkpoint_path}")
    checkpoint = Path(checkpoint_path) if checkpoint_path else None

    base_model_name = args.base_model
    if checkpoint:
        merged_decoder = checkpoint / "decoder_merged"
        if merged_decoder.exists():
            base_model_name = str(merged_decoder)

    model_cfg = ModelConfig(base_model_name=base_model_name, load_in_4bit=args.load_in_4bit)
    vision_cfg = VisionConfig()
    tokenizer, model = build_vlm(model_cfg, vision_cfg, LoraConfigData(enabled=False))
    if not getattr(tokenizer, "chat_template", None):
        try:
            from transformers import AutoTokenizer
            orig_tok = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
            tokenizer.chat_template = orig_tok.chat_template
        except Exception:
            pass

    if checkpoint:
        decoder_adapter = checkpoint / "decoder"
        if decoder_adapter.exists():
            try:
                import peft.import_utils
                peft.import_utils.is_torchao_available = lambda: False
                import peft.tuners.lora.torchao
                peft.tuners.lora.torchao.is_torchao_available = lambda: False
            except Exception:
                pass
            from peft import PeftModel

            model.decoder = PeftModel.from_pretrained(model.decoder, decoder_adapter)
        vision_path = None
        candidates = [checkpoint / "vision_adapter.pt"]
        if checkpoint.name == "merged":
            candidates.append(checkpoint.parent / "vision_adapter.pt")
            candidates.append(checkpoint.parent / "checkpoint-500" / "vision_adapter.pt")
            candidates.append(Path("/content/drive/MyDrive") / checkpoint.parent.name / "checkpoint-500" / "vision_adapter.pt")
            candidates.append(Path("/content/drive/MyDrive") / checkpoint.parent.name / "merged" / "vision_adapter.pt")
        for c in candidates:
            if c.exists():
                vision_path = c
                break
        if not vision_path and args.output_dir:
            latest = find_latest_checkpoint(args.output_dir)
            if latest and (Path(latest) / "vision_adapter.pt").exists():
                vision_path = Path(latest) / "vision_adapter.pt"

        if vision_path and vision_path.exists():
            print(f"Loading vision adapter from: {vision_path}")
            model.load_vision_adapter(vision_path.parent)
        else:
            print(f"Note: No vision_adapter.pt found in {checkpoint}; vision weights remain randomly initialized.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if not args.load_in_4bit and hasattr(model, "to"):
        model.to(device)
    else:
        model.vision_embedder.to(device)
        model.connector.to(device)

    def answer(image, prompt, system_prompt, temperature, max_new_tokens):
        if image is None:
            return "Please upload an image first."
        if not prompt.strip():
            prompt = "Describe this image in detail."
        return generate_answer(
            model=model,
            tokenizer=tokenizer,
            image=image,
            prompt=prompt,
            vision_config=vision_cfg,
            image_token=model_cfg.image_token,
            system_prompt=system_prompt if system_prompt.strip() else None,
            temperature=temperature,
            max_new_tokens=int(max_new_tokens),
        )

    with gr.Blocks(title="Encoder-Free VLM Interactive Demo") as demo:
        gr.Markdown(
            "# 🚀 Encoder-Free VLM (Vision-Language Model)\n"
            "This VLM replaces traditional Vision Encoders (like ViT) with direct RGB patch embeddings mapped into LLM hidden space."
        )
        with gr.Row():
            with gr.Column(scale=1):
                image = gr.Image(type="pil", label="Upload Image")
                prompt = gr.Textbox(label="User Question", value="Describe this image in detail.", lines=2)
                system_prompt = gr.Textbox(
                    label="System Prompt (Optional)",
                    value="You are a helpful and precise vision assistant.",
                    lines=2,
                )
                with gr.Accordion("Advanced Parameters", open=False):
                    temperature = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Temperature")
                    max_new_tokens = gr.Slider(16, 512, value=64, step=16, label="Max New Tokens")
                submit = gr.Button("Submit Question", variant="primary")
            with gr.Column(scale=1):
                output = gr.Textbox(label="VLM Answer", lines=10)

        submit.click(
            answer,
            inputs=[image, prompt, system_prompt, temperature, max_new_tokens],
            outputs=output,
        )

    demo.launch(share=args.share)


if __name__ == "__main__":
    main()


In [ ]:
# Launch the Gradio interface with the trained checkpoint (creates a public share link for Colab).
!python app.py \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --checkpoint-dir outputs/encoder_free_vlm_qwen15b/checkpoint-500 \
  --load-in-4bit \
  --share